# **LLC, грокинг и weight matching: практика**

Тетрадь к модулю «Developmental interpretability». В ней собраны toy-прогоны уроков модуля, и числа в шагах уроков взяты из ее вывода:

* урок «Singular Learning Theory и LLC»: объем почти оптимальных весов $V(\varepsilon)$ и его наклон в логарифмических осях на трех функциях потерь на плоскости и одной в $\mathbb R^3$, оценка LLC динамикой Ланжевена на тех же функциях;
* урок «Фазовые переходы в обучении»: грокинг сложения $(a+b)\bmod 17$ и концентрация Фурье эмбеддинга $C(t)$ по шагам обучения;
* урок «Интерпретируемость пространства весов»: барьер интерполяции до и после weight matching — на переставленной копии сети и на парах независимо обученных сетей; разбиение плоскости двумерной ReLU-сетью на многогранники.

Хватит CPU: все модели маленькие; на трех потоках процессора наш прогон всей тетради занял 3 минуты. Сиды стоят в каждой ячейке, поэтому числа повторяются при повторном запуске. Задания пронумерованы так же, как в шагах урока «LLC, грокинг и weight matching: Практика».

## **Подготовка окружения**
Нужны только torch, numpy, scipy и matplotlib: в Colab они уже стоят. Все считается на CPU, чтобы числа совпали с шагами урока и на машине с GPU.

In [ ]:
import math, json, time
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment

torch.set_default_dtype(torch.float64)
R = {}  # all numbers of the notebook; saved to toys_33_results.json at the end
T0 = time.time()
print("torch", torch.__version__)

## **1. Объем почти оптимальных весов и его наклон**

Три функции потерь на $\mathbb R^2$ с минимумом $L = 0$ в нуле: $w_1^2 + w_2^2$ (теория $\lambda = 1$), $w_1^2 + w_2^4$ ($\lambda = 0.75$) и $w_1^2 w_2^4$ ($\lambda = 1/4$). Объем $V(\varepsilon)$ области $L \le \varepsilon$ внутри квадрата $[-1,1]^2$ — доля из $2^{25} = 33\,554\,432$ равномерных точек, умноженная на площадь квадрата 4. Наклон — метод наименьших квадратов по пяти точкам $\log V$ против $\log \varepsilon$. Та же процедура в кубе $[-1,1]^3$ (объем 8) дает наклон для $w_1^2 + w_2^4 + w_3^6$ из задания урока, где теория $\lambda = 11/12$.

In [ ]:
N_POINTS = 2**25
EPS = [1e-2, 3e-3, 1e-3, 3e-4, 1e-4]
LOSSES = {
    "w1^2 + w2^2": (2, lambda w: w[:, 0]**2 + w[:, 1]**2),
    "w1^2 + w2^4": (2, lambda w: w[:, 0]**2 + w[:, 1]**4),
    "w1^2 * w2^4": (2, lambda w: w[:, 0]**2 * w[:, 1]**4),
    "w1^2 + w2^4 + w3^6": (3, lambda w: w[:, 0]**2 + w[:, 1]**4 + w[:, 2]**6),
}

def volumes(loss, d, seed=0):
    g = torch.Generator().manual_seed(seed)
    w = (torch.rand(N_POINTS, d, generator=g, dtype=torch.float32) * 2 - 1).double()
    L = loss(w)
    return [2**d * (L <= e).double().mean().item() for e in EPS]

def slope(V):
    return float(np.polyfit(np.log(EPS), np.log(V), 1)[0])

R["volume"] = {}
for name, (d, loss) in LOSSES.items():
    V = volumes(loss, d)
    R["volume"][name] = {"V": V, "slope": slope(V)}

print("eps     " + "".join(f"{name:>22}" for name in LOSSES))
for i, e in enumerate(EPS):
    print(f"{e:<8.0e}" + "".join(f"{R['volume'][n]['V'][i]:>22.4g}" for n in LOSSES))
print("slope   " + "".join(f"{R['volume'][n]['slope']:>22.3f}" for n in LOSSES))
print("pi*eps at 1e-2:", round(math.pi * 1e-2, 5), "  exact volume of w1^2*w2^4 at 1e-2:", round(8 * 1e-2**0.25 - 4 * 1e-2**0.5, 4))

**Задание 1. Какой наклон $\log V$ против $\log\varepsilon$ дал прогон для $L = w_1^2 w_2^4$? Ответ дайте с тремя знаками после точки.**

Наклон ниже теоретического $1/4$: точный объем этой области в квадрате равен $8\varepsilon^{1/4} - 4\varepsilon^{1/2}$, и второе слагаемое на этом диапазоне $\varepsilon$ еще заметно.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for name in ["w1^2 + w2^2", "w1^2 + w2^4", "w1^2 * w2^4"]:
    ax.loglog(EPS, R["volume"][name]["V"], "o-", label=f"{name}: slope {R['volume'][name]['slope']:.3f}")
ax.set_xlabel("epsilon"); ax.set_ylabel("V(epsilon)"); ax.legend(); ax.grid(alpha=0.3)
plt.show()

## **2. Оценка LLC динамикой Ланжевена**

Оценщик Lau et al. (2025): $\hat\lambda(w^\star) = n\beta^\ast\big(\mathbb E[L_n(w)] - L_n(w^\star)\big)$, $\beta^\ast = 1/\log n$, среднее — по распределению $\propto \exp\{-n\beta^\ast L_n(w) - \tfrac{\gamma}{2}\|w - w^\star\|^2\}$. Здесь $L_n = L$, $w^\star = 0$, $n = 10^4$, $\gamma = 1$. Выборку строит динамика Ланжевена с полным градиентом:

$$w \leftarrow w - \tfrac{\epsilon}{2}\big(n\beta^\ast \nabla L(w) + \gamma (w - w^\star)\big) + \mathcal N(0, \epsilon I),\qquad \epsilon = 10^{-5},$$

1000 цепочек по 40 000 шагов, первые 10 000 отброшены, сид 0. Около минуты на CPU.

*Статья и библиотека здесь расходятся. У Lau et al. $n\beta^\ast = n/\log n$, где $n$ — размер выборки, а `devinterp` советует брать `default_nbeta(dataloader)` — размер батча, деленный на его логарифм; без явного значения оптимизатор `SGLD` берет `nbeta=1` и предупреждает, что оценка будет неверной. Здесь — как в статье.*

In [ ]:
def llc(loss, d, n=10**4, gamma=1.0, eps=1e-5, chains=1000, steps=40000, burn=10000, seed=0):
    torch.manual_seed(seed)
    nbeta = n / math.log(n)
    w = torch.zeros(chains, d, requires_grad=True)
    total, count = 0.0, 0
    for t in range(steps):
        L = loss(w)
        grad, = torch.autograd.grad(L.sum(), w)
        with torch.no_grad():
            w += -eps / 2 * (nbeta * grad + gamma * w) + math.sqrt(eps) * torch.randn_like(w)
        if t >= burn:
            total += L.detach().mean().item(); count += 1
    return nbeta * total / count  # L(w*) = 0

R["llc"] = {}
for name, (d, loss) in LOSSES.items():
    R["llc"][name] = llc(loss, d)
    print(f"{name:<20} LLC estimate {R['llc'][name]:.3f}   volume slope {R['volume'][name]['slope']:.3f}")

**Задание 2. Какую оценку $\hat\lambda$ дал прогон для $L = w_1^2 + w_2^4$? Ответ дайте с тремя знаками после точки.**

Для суммы степеней по разным координатам среднее $n\beta^\ast L$ по этому распределению равно $\lambda$ при любом $n$, поэтому первые две оценки отличаются от теории только шумом выборки. У $w_1^2 w_2^4$ потеря равна нулю на осях, распределение уходит вдоль них, пока его не остановит привязка $\gamma$, и оценка смещена вниз.

## **3. Грокинг сложения по модулю 17**

Задача — $(a+b)\bmod 17$; вход — склеенные one-hot векторы $a$ и $b$; модель — двухслойный перцептрон ширины 48 с квадратичной активацией $f(x) = \big((xW_1)^2\big)W_2$, веса $W_1$, $W_2$ из $\mathcal N(0,1)$, деленные на корень из числа входов слоя. Ошибка — MSE до one-hot ответа, среднее по всем выходам; полный батч, AdamW со скоростью обучения $3\cdot10^{-3}$ и weight decay 1, 60 000 шагов. В обучении первые 144 пары перестановки `randperm` сида, в тесте остальные 145 из 289; float64, сиды 0–4 (сид задает и разбиение, и инициализацию).

Структурная мера — концентрация Фурье эмбеддинга $a$, то есть строк $W_1$ для one-hot $a$ (матрица $17\times 48$). Вдоль циклической оси $a$ считаем дискретное преобразование Фурье и мощность $P_{nk}$ нейрона $n$ на частоте $k = 1,\dots,8$ (частоты $k$ и $17-k$ сложены, постоянный член отброшен):

$$C(t) = \frac{\sum_n \max_k P_{nk}(t)}{\sum_n\sum_k P_{nk}(t)}.$$

Один прогон — около 15 секунд, десять прогонов (пять сидов при двух скоростях обучения) — две-три минуты.

In [ ]:
P_MOD, WIDTH, STEPS = 17, 48, 60000
A_ = torch.arange(P_MOD).repeat_interleave(P_MOD); B_ = torch.arange(P_MOD).repeat(P_MOD)
X_ADD = torch.cat([torch.nn.functional.one_hot(A_, P_MOD), torch.nn.functional.one_hot(B_, P_MOD)], 1).double()
Y_ADD = (A_ + B_) % P_MOD
T_ADD = torch.nn.functional.one_hot(Y_ADD, P_MOD).double()

def fourier_concentration(W1):
    power = torch.fft.fft(W1[:P_MOD], dim=0).abs()**2   # 17 x 48
    power = power[1:9] + power[9:17].flip(0)               # frequencies k and 17-k together
    return (power.max(0).values.sum() / power.sum()).item()

def grok(seed, lr=3e-3, steps=STEPS, log_every=100):
    torch.manual_seed(seed)
    perm = torch.randperm(P_MOD * P_MOD); tr, te = perm[:144], perm[144:]
    W1 = (torch.randn(2 * P_MOD, WIDTH) / (2 * P_MOD)**0.5).requires_grad_()
    W2 = (torch.randn(WIDTH, P_MOD) / WIDTH**0.5).requires_grad_()
    opt = torch.optim.AdamW([W1, W2], lr=lr, weight_decay=1.0)
    hist, first_train, first_test = [], None, None
    for t in range(steps + 1):
        out = ((X_ADD @ W1)**2) @ W2
        loss = ((out[tr] - T_ADD[tr])**2).mean()
        with torch.no_grad():
            pred = out.argmax(1)
            acc_tr = (pred[tr] == Y_ADD[tr]).double().mean().item()
            acc_te = (pred[te] == Y_ADD[te]).double().mean().item()
            if first_train is None and acc_tr == 1: first_train = t
            if first_test is None and acc_te == 1: first_test = t
            if t % log_every == 0:
                hist.append([t, acc_tr, acc_te, loss.item(), fourier_concentration(W1)])
        if t == steps: break
        opt.zero_grad(); loss.backward(); opt.step()
    return {"hist": hist, "first_train": first_train, "first_test": first_test}

R["grok"] = {str(s): grok(s) for s in range(5)}
print("seed 0:")
print(f"{'step':>6} {'acc train':>10} {'acc test':>9} {'MSE train':>10} {'C(t)':>6}")
for t, a_tr, a_te, mse, c in R["grok"]["0"]["hist"]:
    if t in (0, 500, 5000, 10000, 20000, 60000):
        print(f"{t:>6} {a_tr:>10.3f} {a_te:>9.3f} {mse:>10.1e} {c:>6.3f}")
print()
for s in range(5):
    g = R["grok"][str(s)]
    print(f"seed {s}: train accuracy 1.000 from step {g['first_train']}, test accuracy 1.000 from step {g['first_test']}, C at step 60000 = {g['hist'][-1][4]:.3f}")

**Задание 3. С какого шага у сида 0 точность на тесте равна 1.000?**

**Задание 4. На каком отрезке лежит $C$ на шаге 60 000 у пяти сидов?**

In [ ]:
h = np.array(R["grok"]["0"]["hist"])
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(h[:, 0], h[:, 1], label="train accuracy"); ax.plot(h[:, 0], h[:, 2], label="test accuracy"); ax.plot(h[:, 0], h[:, 4], label="Fourier concentration C(t)")
ax.set_xscale("symlog", linthresh=100); ax.set_xlabel("training step"); ax.legend(); ax.grid(alpha=0.3)
plt.show()

Та же постановка со скоростью обучения $10^{-3}$: длина каждой фазы грокинга зависит от скорости обучения и weight decay.

In [ ]:
R["grok_lr1e-3"] = {}
for s in range(5):
    g = grok(s, lr=1e-3, log_every=STEPS)
    R["grok_lr1e-3"][str(s)] = {"first_train": g["first_train"], "first_test": g["first_test"]}
    print(f"lr 1e-3, seed {s}: train accuracy 1.000 from step {g['first_train']}, test accuracy 1.000 from step {g['first_test']}")
print("seeds with test accuracy 1.000 within 60000 steps:", sum(v["first_test"] is not None for v in R["grok_lr1e-3"].values()), "of 5")

## **4. Барьер интерполяции до и после weight matching**

Сеть $f(x) = W_2\,\mathrm{ReLU}(W_1 x + b_1) + b_2$. Путь между моделями — $\theta(\alpha) = \alpha\,\theta_A + (1-\alpha)\,\theta_B$, так что $\alpha = 0$ — модель $B$, $\alpha = 1$ — модель $A$. Барьер по Entezari et al. — наибольшее по точкам $\alpha$ превышение потерь над линейной интерполяцией потерь концов. Weight matching для одного скрытого слоя ищет перестановку нейронов $B$, при которой сумма скалярных произведений весов соответствующих нейронов ($W_1$, $b_1$ и столбцов $W_2$) наибольшая: это задача о назначениях, ее решает `linear_sum_assignment`.

**Опыт 1: переставленная копия.** Вход размерности 4, $h = 6$ скрытых нейронов, выход размерности 2, все веса $A$ из $\mathcal N(0,1)$, сид 0. Нейрон $i$ модели $B$ — нейрон $P_{\text{true}}(i)$ модели $A$, $P_{\text{true}} = (2,4,1,3,5,0)$. Цель — выход $A$ на 1000 гауссовых входах, потери — MSE, поэтому $L(\theta_A) = L(\theta_B) = 0$.

In [ ]:
def net(th, x):
    W1, b1, W2, b2 = th
    return torch.relu(x @ W1.T + b1) @ W2.T + b2

def path(a, b, alpha):  # alpha = 1 -> model a, alpha = 0 -> model b
    return [alpha * u + (1 - alpha) * v for u, v in zip(a, b)]

def permute(th, P):
    W1, b1, W2, b2 = th
    return [W1[P], b1[P], W2[:, P], b2]

def weight_matching(a, b):
    cost = a[0] @ b[0].T + torch.outer(a[1], b[1]) + a[2].T @ b[2]
    rows, cols = linear_sum_assignment(cost.numpy(), maximize=True)
    return torch.as_tensor(cols)

torch.manual_seed(0)
model_A = [torch.randn(6, 4), torch.randn(6), torch.randn(2, 6), torch.randn(2)]
P_TRUE = torch.tensor([2, 4, 1, 3, 5, 0])
model_B = permute(model_A, P_TRUE)
X1 = torch.randn(1000, 4); Y1 = net(model_A, X1)
mse1 = lambda th: ((net(th, X1) - Y1)**2).mean().item()

ALPHAS1 = [0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0]
R["exp1"] = {"max_abs_diff_fA_fB": (net(model_A, X1) - net(model_B, X1)).abs().max().item(),
             "alphas": ALPHAS1, "mse_before": [mse1(path(model_A, model_B, a)) for a in ALPHAS1]}
pi = weight_matching(model_A, model_B)
model_B_aligned = permute(model_B, pi)
R["exp1"]["matching"] = pi.tolist()
R["exp1"]["max_abs_W1_diff_after"] = (model_A[0] - model_B_aligned[0]).abs().max().item()
R["exp1"]["max_mse_after"] = max(mse1(path(model_A, model_B_aligned, a)) for a in np.linspace(0, 1, 11))
print("max |f_A - f_B| on the inputs:", f"{R['exp1']['max_abs_diff_fA_fB']:.1e}")
print("alpha:            ", "  ".join(f"{a:>6.1f}" for a in ALPHAS1))
print("MSE, no alignment:", "  ".join(f"{m:>6.3f}" for m in R["exp1"]["mse_before"]))
print("weight matching permutation:", R["exp1"]["matching"])
print("max |W1_A - W1_B'| after matching:", R["exp1"]["max_abs_W1_diff_after"], "  max MSE on the path after matching:", f"{R['exp1']['max_mse_after']:.1e}")

**Опыт 2: две независимо обученные сети.** Сеть-учитель $8\to32\to1$ (веса и смещения из $\mathcal N(0,1)$, деленные на корень из числа входов слоя, сид 1) задает цель; ученики $8\to h\to 1$ со стандартной инициализацией `nn.Linear` обучаются с разных сидов: Adam, скорость обучения $10^{-3}$, 3000 шагов, батч 256, 4096 обучающих входов. Потери — MSE на 4096 тестовых входах, барьер — 11 точек $\alpha$, пять пар моделей (сиды 100 и 101, 102 и 103, …, 108 и 109). Три значения $h$ — около минуты на CPU.

In [ ]:
torch.manual_seed(1)
teacher = [torch.randn(32, 8) / 8**0.5, torch.randn(32) / 8**0.5, torch.randn(1, 32) / 32**0.5, torch.randn(1) / 32**0.5]
g = torch.Generator().manual_seed(2)
X_train = torch.randn(4096, 8, generator=g); X_test = torch.randn(4096, 8, generator=g)
Y_train, Y_test = net(teacher, X_train), net(teacher, X_test)
ALPHAS2 = np.linspace(0, 1, 11)

def student(h, seed):
    torch.manual_seed(seed)
    l1, l2 = torch.nn.Linear(8, h), torch.nn.Linear(h, 1)
    th = [p.data.clone().requires_grad_() for p in (l1.weight, l1.bias, l2.weight, l2.bias)]
    opt = torch.optim.Adam(th, lr=1e-3)
    g = torch.Generator().manual_seed(seed)
    for _ in range(3000):
        idx = torch.randint(0, 4096, (256,), generator=g)
        loss = ((net(th, X_train[idx]) - Y_train[idx])**2).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    return [p.detach() for p in th]

test_mse = lambda th: ((net(th, X_test) - Y_test)**2).mean().item()

def excess(a, b):  # loss on the path minus the linear interpolation of the endpoint losses
    la, lb = test_mse(a), test_mse(b)
    return [test_mse(path(a, b, al)) - (al * la + (1 - al) * lb) for al in ALPHAS2]

R["exp2"] = {}
print(f"{'h':>5} {'MSE A':>7} {'MSE B':>7}   barrier, no alignment     barrier after matching    ratio")
for h in [16, 128, 1024]:
    pairs = []
    for k in range(5):
        a, b = student(h, 100 + 2 * k), student(h, 101 + 2 * k)
        b_aligned = permute(b, weight_matching(a, b))
        pairs.append({"mse_A": test_mse(a), "mse_B": test_mse(b), "excess_before": excess(a, b), "excess_after": excess(a, b_aligned)})
    before = np.array([max(p["excess_before"]) for p in pairs]); after = np.array([max(p["excess_after"]) for p in pairs])
    R["exp2"][str(h)] = {"pairs": pairs, "barrier_before": before.tolist(), "barrier_after": after.tolist()}
    print(f"{h:>5} {np.mean([p['mse_A'] for p in pairs]):>7.4f} {np.mean([p['mse_B'] for p in pairs]):>7.4f}"
          f"   {before.mean():.4f} [{before.min():.4f}, {before.max():.4f}]   {after.mean():.4f} [{after.min():.4f}, {after.max():.4f}]   {before.mean() / after.mean():.1f}")

**Задание 5. Во сколько раз weight matching уменьшает средний барьер при $h = 16$? Ответ дайте с одним знаком после точки.**

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for h, c in [("16", "C0"), ("128", "C1"), ("1024", "C2")]:
    pairs = R["exp2"][h]["pairs"]
    ax.plot(ALPHAS2, np.mean([p["excess_before"] for p in pairs], 0), "--", color=c, label=f"h={h}, no alignment")
    ax.plot(ALPHAS2, np.mean([p["excess_after"] for p in pairs], 0), "-", color=c, label=f"h={h}, weight matching")
ax.set_xlabel("alpha (0 = model B, 1 = model A)"); ax.set_ylabel("excess test MSE"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.show()

## **5. Многогранники двумерной ReLU-сети**

Сеть $2\to8\to8$ с ReLU, веса и смещения из $\mathcal N(0,1)$, веса второго слоя, деленные на $\sqrt 8$, сид 0. Сплайн-код входа — 16 бит: какие нейроны обоих слоев выше порога. На сетке $801\times801$ точек в квадрате $[-3,3]^2$ считаем, сколько разных кодов встречается: это число многогранников, в которые попала хотя бы одна точка сетки. Нейроны первого слоя делят плоскость прямыми $w_i^\top x + b_i = 0$; нейроны второго слоя дробят эти области дальше по ломаным границам. Перестановка нейронов переставляет биты кода и не меняет ни одного многогранника.

In [ ]:
torch.manual_seed(0)
poly = {"W1": torch.randn(8, 2), "b1": torch.randn(8), "W2": torch.randn(8, 8) / 8**0.5, "b2": torch.randn(8)}
xs = torch.linspace(-3, 3, 801)
grid = torch.stack(torch.meshgrid(xs, xs, indexing="xy"), -1).reshape(-1, 2)

def spline_code(th, x):
    z1 = x @ th["W1"].T + th["b1"]
    z2 = torch.relu(z1) @ th["W2"].T + th["b2"]
    return torch.cat([z1 > 0, z2 > 0], 1)

def n_regions(bits):
    return len(torch.unique(bits.long() @ (2 ** torch.arange(bits.shape[1])), sorted=False))

codes = spline_code(poly, grid)
perm = torch.tensor([3, 0, 6, 1, 7, 2, 5, 4])
poly_perm = {"W1": poly["W1"][perm], "b1": poly["b1"][perm], "W2": poly["W2"][:, perm], "b2": poly["b2"]}
R["poly"] = {"weights": {k: v.tolist() for k, v in poly.items()},
             "regions_layer1": n_regions(codes[:, :8]), "regions_both": n_regions(codes), "regions_both_permuted": n_regions(spline_code(poly_perm, grid))}
print("regions cut by the 8 first-layer lines:", R["poly"]["regions_layer1"])
print("regions of the full 16-bit code:", R["poly"]["regions_both"], "  after permuting first-layer neurons:", R["poly"]["regions_both_permuted"])

**Задание 6. Сколько многогранников полного 16-битного кода попало на сетку?**

In [ ]:
key = (codes.long() @ (2 ** torch.arange(16))).reshape(801, 801)
_, ids = torch.unique(key, return_inverse=True)
rng = np.random.default_rng(0)
colors = rng.random((ids.max().item() + 1, 3)) * 0.6 + 0.4
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(colors[ids.numpy()], extent=[-3, 3, -3, 3], origin="lower")
t = np.linspace(-3, 3, 2)
for (w_x, w_y), b in zip(poly["W1"].tolist(), poly["b1"].tolist()):
    ax.plot(t, -(w_x * t + b) / w_y, "k-", lw=0.8)
ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_title("polytopes of the 2-8-8 ReLU network")
plt.show()

json.dump(R, open("toys_33_results.json", "w"))
print("saved toys_33_results.json; total time", round(time.time() - T0), "s")

## **Итог**

* Наклон объема почти оптимальных весов падает с вырожденностью: теоретически 1, 0.75 и 1/4 при одном и том же $d = 2$; оценщик LLC повторяет это на регулярной и квазиоднородной потере и смещен вниз там, где потеря равна нулю на целых осях.
* При грокинге точность на обучении равна 1 уже через сотню шагов, а точность на тесте и концентрация Фурье растут вместе на тысячах шагов после этого.
* Высокий барьер между двумя сетями бывает следствием одной перестановки нейронов; weight matching убирает его целиком у переставленной копии и уменьшает в 17–98 раз у независимо обученных сетей.